In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from pandas import DataFrame
from icecream import ic

from src.utils.infer import infer_dtypes
from src.utils.paths import DATA_PATH

EXTERNAL_PATH = DATA_PATH / "external"
filename = "TB_CEP_BR_2018.csv"
filepath = EXTERNAL_PATH / filename

if not filepath.exists():
    raise FileNotFoundError

In [ ]:
df = pd.read_csv(
    filepath,
    delimiter=";",
    dtype=str,
    encoding="utf_8",
)
df.shape

In [ ]:
# scrap
# idx_headers = df[df[0].str.contains("CEP")].index.tolist()

# # table a
# df_a_headers: list[str] = df.iloc[idx_headers[0]].tolist()

# df_a = df.iloc[idx_headers[0] + 1 : idx_headers[1]].copy().reset_index(drop=True)
# df_a.columns = df_a_headers

# # table b
# df_b_headers: list[str] = df.iloc[idx_headers[1]].tolist()
# df_b_headers[0] = 'CEP'
# df_b_headers.pop(-1)

# df_b = df.iloc[idx_headers[1]+1:].copy().reset_index(drop=True)
# # df_b.columns = df_b_headers

# df_b

# # df_a first table, df_b second table. indexes reset

In [ ]:
# this file has two tables stacked one on top of the other
# define table A as the table that appears with columns:
#   CEC, UF, CIDADE, BAIRRO, LOGRADOURO, COMPLEMENTO
# define table B as the table that appears second with columns:
#   CEC, UF, CIDADE, BAIRRO, LOGRADOURO

# tables A and B have an intersection
# prioritise table A
#   natural to assume priority since first
#   uses the additional column
# any CEP not in A but in B, include with A
# have a separate table B

# want to make
#   single reference table: (on CEP)
#       A u B\A (prefer A, supplement with unique B)
#   extra table
#       B (whole of B)

# partition
idx_repeat_header = df[df["CEP"].str.contains("CEP")].index[0]
df_A = df.iloc[:idx_repeat_header, :].copy()
df_B = df.iloc[idx_repeat_header + 1 :, :-1].copy().reset_index(drop=True)

df.shape[0] == df_A.shape[0] + df_B.shape[0] + 1
# the +1 is the row of the extra headers

In [ ]:
# exclusive or (xor) operator ^
# distinct elements in A, union, distinct elements in B
# symmetric difference
# A\B u B\a
symm_diff = set(df_A["CEP"]) ^ set(df_B["CEP"])

# on col CEP, A\B
cep_AmB = set(df_A["CEP"]) - set(df_B["CEP"])

# on col CEP B\A
cep_BmA = set(df_B["CEP"]) - set(df_A["CEP"])

In [ ]:
# unique CEPs on table B
cep_BmA_list = list(sorted(cep_BmA))
df_BmA = df[df["CEP"].isin(cep_BmA_list)]
# infer_dtypes(df_BmA)

# AuBmA means A u (B\A) means A union (B set minus A)
df_AuBmA = pd.concat([df_A, df_BmA])

In [ ]:
# then I have two tables: df_AuBmA unique CEP, and df_B unique CEP

# TB_CEP_BR_2018__AuBmA.csv
df_AuBmA_attrs = infer_dtypes(df_AuBmA)

# TB_CEP_BR_2018__B.csv
df_B_attrs = infer_dtypes(df_B)

In [ ]:
# finding sql server data types:
pd.set_option("display.max_columns", None)

# df_AuBmA_attrs
"""
CEP
    unique
    no nulls
    max 8
    fixed length
UF
    no nulls
    max 14
CIDADE
    max 62
    nulls
    nonascii
BAIRRO
    max 65
    nulls
    nonascii
LOGRADOURO
    max 132
    nulls
    nonascii
COMPLEMENTO
    max 78
    nulls
    nonascii
"""

# df_B_attrs
"""
no nulls
CEP
    max 8
    unique
    no nulls
    fixed lengths
UF
    max 2
    no nulls
    fixed lengths
CIDADE
    max 23
    no null
    nonascii
BAIRRO
    max 63
    no nulls
    nonascii
LOGRADOURO
    max 71
    no nulls
    nonascii
"""
print()

In [ ]:
# df without repeat header
# mrh means minus repeat headers
df_mrh = df[~df["CEP"].str.contains("CEP")]
df_mrh_attrs = infer_dtypes(df_mrh)

In [ ]:
"""
no nulls: cep, uf, cidade

CEP char(8) not null
UF varchar(20) not null
    max 14
CIDADE nvarchar(70) not null
    max 62
    nonascii
BAIRRO nvarchar(70)
    max 65
    nonascii
LOGRADOURO nvarchar(150)
    max 132
    nonascii
COMPLEMENTO nvarchar(100)
    max 78
    nonascii
"""
df_mrh_attrs

In [ ]:
print(df_A.shape[0])
print(df_B.shape[0])
df_A

In [ ]:
df_B

In [ ]:
repeat_cep = df_mrh[df_mrh["CEP"].duplicated()]["CEP"].tolist()
len(repeat_cep)

In [ ]:
start = 300000
diff = 50

end = 10

# idxs = repeat_cep[start:end]
idxs = repeat_cep[start : start + diff]
for i in idxs:
    print(df_mrh[df_mrh["CEP"] == i])